# 00.05 · First contact: Groq + LangChain

**Module 00 · Foundations, Tooling & Research Skills** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 75–90 min |
| 📋 Prerequisites | Lessons 00.01–00.04, the `lc-mastery` kernel, and `GROQ_API_KEY` in `.env` (Lesson 00.04). |
| 🔑 Keys | `GROQ_API_KEY` (optional: without it, API cells print ⏭️ Skipped) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-11 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**.
> Run cells **from top to bottom**. If anything looks strange, use **Kernel → Restart Kernel and Run All Cells…**

## Learning objectives

1. **Install** `langchain` and `langchain-groq` into the course environment with `%pip`, and verify the versions.
2. **List** the models that are live on your Groq account through the Models API (`GET /openai/v1/models`).
3. **Call** `openai/gpt-oss-20b` with `ChatGroq` and with `init_chat_model("groq:...")`, and explain when to use each.
4. **Read** the returned `AIMessage`: `.text`, reasoning content, `response_metadata`, and `usage_metadata` (tokens), and turn tokens into cost.
5. **Handle** the common errors (missing key, invalid key, retired model, rate limit) with clear messages.

## 📚 Official reading (verified 2026-10-11)

| Page | Why |
|---|---|
| [ChatGroq integration](https://docs.langchain.com/oss/python/integrations/chat/groq) · [Groq provider page](https://docs.langchain.com/oss/python/integrations/providers/groq) | Install, credentials, instantiation, invocation |
| [Models (LangChain)](https://docs.langchain.com/oss/python/langchain/models) | `init_chat_model`, parameters, token usage |
| [`init_chat_model` reference](https://reference.langchain.com/python/langchain/chat_models/base/init_chat_model) | Provider prefixes, e.g. `groq` → `langchain-groq` |
| [Groq quickstart](https://console.groq.com/docs/quickstart) · [Groq models and the Models API](https://console.groq.com/docs/models) · [API reference: list models](https://console.groq.com/docs/api-reference#models-list) | The raw API underneath |
| [Groq reasoning](https://console.groq.com/docs/reasoning) · [Groq errors](https://console.groq.com/docs/errors) | `reasoning_effort`, `include_reasoning`, status codes |

> **Heads-up:** the ChatGroq page's example uses `qwen/qwen3-32b`, which Groq shut down on 2026-07-17. So we take the *code shape* from the docs and the *model ID* from `.env` (`MASTERY_MODEL_FAST`).

---
## Step 0 · Confirm the kernel

Packages are about to be installed into **whatever environment this kernel runs**. Make sure it is `lc-mastery`, not `base`.

In [ ]:
import sys
from pathlib import Path

print("Kernel Python:", sys.executable)
in_course_env = Path(sys.prefix).name == "lc-mastery" or (Path(sys.prefix) / "pyvenv.cfg").is_file()
print("✅ Course environment." if in_course_env else
      "⚠️ Not lc-mastery! Kernel → Change Kernel… → Python (lc-mastery) before installing anything.")

## Step 1 · Install the packages with `%pip`

`%pip install` runs `pip` with **this kernel's Python**, so the packages land in `lc-mastery` (Lesson 00.02). Version ranges come from [`requirements.txt`](../../requirements.txt) (verified on PyPI on 2026-10-11): at least the tested version, below the next major version.

`pandas` is used for tables here and in Project 00.

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-groq>=1.1.3,<2" "pandas>=3.0.6,<4"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."* (IPython prints it after **every** `%pip`), possibly below a few pip lines. The LangChain packages are already installed by `pip install -e ".[dev]"`; this cell mainly adds `pandas`. Restart only if a package was upgraded after you had imported it.

> **When to restart:** if a package was *upgraded* and you had already imported it in this kernel, Python keeps the old version in memory. Then use **Kernel → Restart Kernel…** and run the notebook from the top. On a first install, no restart is needed.

**If the install fails:** read the last lines. `Could not find a version that satisfies the requirement` usually means an old Python (check Step 0); SSL or proxy errors mean a network problem.

In [ ]:
from importlib import metadata

for package in ("langchain", "langchain-core", "langchain-groq", "groq", "pandas"):
    print(f"{package:<15} {metadata.version(package)}")

**You should now see** `langchain 1.4.x` (or newer 1.x), `langchain-core 1.6.x`, `langchain-groq 1.1.x`, and `groq` (Groq's own SDK, installed automatically as a dependency of `langchain-groq`).

## Step 2 · Load settings and check the key

In [ ]:
# Setup 1: Find the repository root.
# WHY: Jupyter starts this kernel in the notebook's own folder (modules/00-foundations/),
# but .env, data/ and src/ live at the repo root, two levels up.
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError(
        "Could not find the course repo (pyproject.toml). "
        "Start Jupyter from inside your clone of langchain-langgraph-mastery (or set LC_MASTERY_ROOT to it)."
    )

# Setup 2: Import the course package `mastery` from THIS clone's src/.
# WHY: pytest imports it from src/ too (pythonpath = ["src"] in pyproject.toml), so notebooks
# and tests run the same code even if this env was installed from another copy of the repo.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")  # The env's own install; terminal commands use it.
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root :", REPO_ROOT)
print("mastery   :", mastery.__version__, "from", mastery.__file__)

# Setup 3: Check that the env's own install (create-course-env guide, Step 5) points here too.
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder.")
    print("   Kernel → Restart Kernel, then run the notebook from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
from mastery.config import MissingSettingError, load_env, load_settings, require_env

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded above.

try:
    require_env("GROQ_API_KEY", dotenv_path=dotenv_path)
    HAVE_GROQ_KEY = True
    print("✅ GROQ_API_KEY is set.")
except MissingSettingError as error:
    HAVE_GROQ_KEY = False
    print("⚠️ " + str(error))
    print("\nThe API cells below will be SKIPPED with a message. Everything else still runs.")

print("\nFast model (MASTERY_MODEL_FAST)          :", settings.model_fast)
print("Reasoning model (MASTERY_MODEL_REASONING):", settings.model_reasoning)

> **Pattern used in every API cell from now on:** `if not HAVE_GROQ_KEY: print(skip message)` / `else: call the API`. A missing key never crashes *Restart & Run All*; it tells you exactly what was skipped.

---
## Part 1 · Which models are live on *your* account?

Before calling a model, ask Groq which model IDs exist. `mastery.groq_catalog.fetch_groq_models()` sends:

```http
GET https://api.groq.com/openai/v1/models
Authorization: Bearer <GROQ_API_KEY>
```

and turns the JSON (`{"object": "list", "data": [...]}`) into tidy `GroqModel` objects ([API reference](https://console.groq.com/docs/api-reference#models-list)). It's about 230 lines of commented code you can read in [`src/mastery/groq_catalog.py`](../../src/mastery/groq_catalog.py). Listing models **does not generate tokens**, so it doesn't use your token quota.

In [ ]:
# 🌐 Calls the Groq Models API (no tokens generated).
import pandas as pd
from IPython.display import display

from mastery.groq_catalog import GroqAPIError, fetch_groq_models, looks_like_chat_model

live_models = []
if not HAVE_GROQ_KEY:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see Step 2).")
else:
    try:
        live_models = fetch_groq_models()
    except GroqAPIError as error:
        print("⚠️", error)

if live_models:
    table = pd.DataFrame(
        [{"id": m.id, "owned_by": m.owned_by, "context_window": m.context_window,
          "chat model?": looks_like_chat_model(m.id)} for m in live_models]
    )
    display(table)

In [ ]:
# Are the models configured in .env really live? (This check runs only if Part 1 succeeded.)
if live_models:
    live_ids = {m.id for m in live_models}
    for label, model_id in (("fast", settings.model_fast), ("reasoning", settings.model_reasoning)):
        status = "✅ live" if model_id in live_ids else "❌ NOT in your live list: check Groq deprecations"
        print(f"{label:<10} {model_id:<24} {status}")
else:
    print("⏭️ Skipped (no live model list).")

**You should now see** a table with about 10 rows including `openai/gpt-oss-20b`, `openai/gpt-oss-120b`, `qwen/qwen3.8-27b`, Whisper, Orpheus, and the safety models, and both configured models marked ✅ live.

The **"chat model?"** column is a heuristic (speech and safety models are not general chat models). Always confirm on the [models page](https://console.groq.com/docs/models).

---
## Part 2 · Your first call with `ChatGroq`

`ChatGroq` is LangChain's chat-model class for Groq. Creating it does not call the API yet; it reads `GROQ_API_KEY` from the environment and stores it as a `SecretStr` (printed as `**********`).

Parameters we set on purpose:

| Parameter | Value | Why |
|---|---|---|
| `model` | `settings.model_fast` | From `.env`, never hard-coded |
| `temperature` | `0` | More repeatable answers while learning |
| `reasoning_effort` | `"low"` | `gpt-oss` models reason before answering; `low` keeps it fast and cheap ([Groq reasoning](https://console.groq.com/docs/reasoning)) |
| `max_retries` | `2` | Retry transient errors (the default is also 2) |
| `timeout` | `60` | Seconds before giving up instead of hanging |

In [ ]:
from langchain_groq import ChatGroq

llm = None
if not HAVE_GROQ_KEY:
    print("⏭️ Skipped: ChatGroq needs GROQ_API_KEY even to be created.")
else:
    llm = ChatGroq(
        model=settings.model_fast,
        temperature=0,
        reasoning_effort="low",
        max_retries=2,
        timeout=60,
    )
    print("Model object:", type(llm).__name__, "→", llm.model_name)
    print("Key stored as:", repr(llm.groq_api_key))   # SecretStr('**********'): safe to print.

Now the call. A chat model takes a **list of messages**. The simplest format is `(role, text)` tuples: `"system"` sets behavior, `"human"` is the user.

In [ ]:
%%time
# 💸 Calls Groq (openai/gpt-oss-20b by default), ~150–400 tokens. Free on the free plan.
messages = [
    ("system", "You are a concise assistant for a LangChain course. Answer in at most two sentences."),
    ("human", "What is the difference between a framework and a runtime for AI agents?"),
]

response = None
if not HAVE_GROQ_KEY or llm is None:
    print("⏭️ Skipped: no model (GROQ_API_KEY not set).")
else:
    response = llm.invoke(messages)
    print(response.text)

**You should now see** a two-sentence answer, followed by `CPU times` and `Wall time` from `%%time`. Wall time is typically **well under 2 seconds** on Groq; CPU time is tiny because your computer only waited for the network (Lesson 00.03).

---
## Part 3 · Anatomy of an `AIMessage`

`invoke` returns an `AIMessage`. Learn its parts now; every later module uses them.

In [ ]:
if response is None:
    print("⏭️ Skipped: no response.")
else:
    print("Type                :", type(response).__name__)
    print("text (the answer)   :", response.text[:120], "…")
    reasoning = response.additional_kwargs.get("reasoning_content", "")
    # gpt-oss models return their reasoning separately; langchain-groq puts it here.
    print("reasoning (preview) :", (reasoning[:120] + " …") if reasoning else "<none returned>")
    print("model_name          :", response.response_metadata.get("model_name"))
    print("finish_reason       :", response.response_metadata.get("finish_reason"))
    print("usage_metadata      :", response.usage_metadata)

| Field | What it is | Used in |
|---|---|---|
| `.text` | The answer as plain text | everywhere |
| `.content` | Raw content (a string, or a list of content blocks for multimodal output) | Module 01, 13 |
| `additional_kwargs["reasoning_content"]` | The model's reasoning, returned separately for `gpt-oss` (Groq's `reasoning` field) | debugging |
| `response_metadata` | Provider details: `model_name`, `finish_reason` (`stop` = finished normally, `length` = cut off), timings | monitoring |
| `usage_metadata` | Standard token counts: `input_tokens`, `output_tokens`, `total_tokens`, and details such as `output_token_details["reasoning"]` | cost tracking |

`usage_metadata` has the **same keys for every provider** in LangChain ([Models → Token usage](https://docs.langchain.com/oss/python/langchain/models)). That is what makes cost tracking portable.

Providers price tokens **per million**, with one price for input (what you send) and a higher one for output (what the model writes, *including* its hidden reasoning tokens):

**cost = input tokens / 1,000,000 × input price + output tokens / 1,000,000 × output price**

Let's price this call:

In [ ]:
# VERIFIED 2026-10-11: Groq list prices per 1M tokens (console.groq.com/docs/models).
GROQ_PRICES = {"openai/gpt-oss-20b": (0.075, 0.30), "openai/gpt-oss-120b": (0.15, 0.60)}

if response is None or response.usage_metadata is None:
    print("⏭️ Skipped: no usage data.")
else:
    usage = response.usage_metadata
    price_in, price_out = GROQ_PRICES.get(settings.model_fast, (0.0, 0.0))
    cost = usage["input_tokens"] / 1e6 * price_in + usage["output_tokens"] / 1e6 * price_out
    reasoning_tokens = usage.get("output_token_details", {}).get("reasoning", 0)
    print(f"Tokens: {usage['input_tokens']} in, {usage['output_tokens']} out ({reasoning_tokens} of them reasoning)")
    print(f"Paid-tier cost of this call: ${cost:.6f}  (free on the free plan)")

**You should now see** the token counts of your call (a few hundred in total, part of them reasoning) and a paid-tier cost far below one cent. **Write down your `output_tokens`:** Lesson 00.07 uses that number to estimate what a whole Project 00 run costs and how many calls the free tier allows.

---
## Part 4 · The same call with `init_chat_model`

`init_chat_model("provider:model")` creates the right chat-model class from a string. The prefix `groq` maps to the `langchain-groq` package ([reference](https://reference.langchain.com/python/langchain/chat_models/base/init_chat_model)). Note the model ID itself contains a slash: `groq:openai/gpt-oss-20b` means *provider* `groq`, *model* `openai/gpt-oss-20b` (OpenAI's open-weight model, served by Groq).

In [ ]:
from langchain.chat_models import init_chat_model

llm2 = None
if not HAVE_GROQ_KEY:
    print("⏭️ Skipped: GROQ_API_KEY not set.")
else:
    model_string = f"groq:{settings.model_fast}"        # Built from config, not hard-coded.
    llm2 = init_chat_model(model_string, temperature=0, reasoning_effort="low", timeout=60)
    print(model_string, "→", type(llm2).__name__)       # ChatGroq: the same class as Part 2.

In [ ]:
# 💸 Calls Groq, ~100–300 tokens.
from langchain.messages import HumanMessage, SystemMessage

if not HAVE_GROQ_KEY or llm2 is None:
    print("⏭️ Skipped: no model.")
else:
    # Message objects instead of tuples: same meaning, more explicit, and they carry metadata.
    reply = llm2.invoke([
        SystemMessage("Answer in one short sentence."),
        HumanMessage("Why should model IDs live in configuration instead of code?"),
    ])
    print(reply.text)

### `ChatGroq` or `init_chat_model`?

| Use | When |
|---|---|
| `init_chat_model("groq:…")` | **Default in this course.** One string switches providers (`openai:gpt-6-luna`, `google_genai:gemini-3.8-flash`…), which is how `mastery.models` (Module 01) and the paid benchmark work |
| `ChatGroq(...)` | You need a Groq-specific option and want your editor's autocomplete and type checking for it |

Both return the same `ChatGroq` object here.

### Streaming, in one cell

`stream()` yields the answer in chunks as it is generated, which is how chat UIs show text "typing" (Module 01 goes deeper).

In [ ]:
# 💸 Calls Groq, ~100–300 tokens.
if not HAVE_GROQ_KEY or llm2 is None:
    print("⏭️ Skipped: no model.")
else:
    for chunk in llm2.stream("Count from 1 to 5, separated by spaces."):
        print(chunk.text, end="", flush=True)
    print()

---
## Part 5 · When things go wrong

The four errors you will meet most, and what they mean ([Groq errors](https://console.groq.com/docs/errors)):

| Error (Python class from the `groq` SDK) | HTTP | Meaning | Fix |
|---|---|---|---|
| `GroqError: The api_key client option must be set…` | — | No key in the environment | Lesson 00.04; restart the kernel |
| `AuthenticationError` | 401 | Key invalid or deleted | Create a new key; update `.env`; restart |
| `NotFoundError` / `BadRequestError` mentioning the model | 404 / 400 | Model ID wrong or retired | Check the live list (Part 1) and [deprecations](https://console.groq.com/docs/deprecations) |
| `RateLimitError` | 429 | Over your RPM/TPM/RPD/TPD | Wait `retry-after` seconds; reduce tokens (Lesson 00.07 explains the limits) |

The next cell triggers a **401 on purpose** with an obviously fake key. It needs internet but **no real key**, and costs nothing, so it runs for everyone.

In [ ]:
import groq


def explain_groq_error(error: Exception) -> str:
    """Turn a Groq SDK exception into a one-line, actionable explanation (never includes the key)."""
    if isinstance(error, groq.AuthenticationError):
        return "401 Invalid API key → create a new key at https://console.groq.com/keys, update .env, restart the kernel."
    if isinstance(error, groq.RateLimitError):
        return "429 Rate limit → wait for the 'retry-after' seconds; see https://console.groq.com/docs/rate-limits"
    if isinstance(error, (groq.NotFoundError, groq.BadRequestError)):
        return f"{error.status_code} Bad request or unknown model → check the model ID against your live list (Part 1)."
    if isinstance(error, groq.APIConnectionError):
        return "Network problem → check your connection, VPN, or proxy."
    return f"{type(error).__name__}: {error}"


# A deliberately FAKE key: Groq rejects it with 401. No tokens, no cost.
fake_llm = ChatGroq(model=settings.model_fast, api_key="gsk_FAKE_key_for_error_demo", max_retries=0, timeout=20)
try:
    fake_llm.invoke("hello")
except groq.APIError as error:   # Base class of the Groq SDK's API errors.
    print("Caught:", type(error).__name__)
    print("Explanation:", explain_groq_error(error))

In [ ]:
# 💸 Calls Groq with a RETIRED model ID on purpose (fails fast; no tokens are generated).
RETIRED_ID = "llama-3.3-70b-versatile"   # VERIFIED 2026-10-11: shut down 2026-08-16 for free/developer tiers.

if not HAVE_GROQ_KEY:
    print("⏭️ Skipped: needs a real GROQ_API_KEY to show the 'retired model' error.")
else:
    try:
        ChatGroq(model=RETIRED_ID, max_retries=0, timeout=20).invoke("hello")
        print(f"{RETIRED_ID} answered. Your organization may have an enterprise contract that still includes it.")
    except groq.APIError as error:
        print("Caught:", type(error).__name__, "→", explain_groq_error(error))
        print("Groq said:", str(error)[:200])

### What just happened?

- The fake key produced `AuthenticationError` (401), caught and explained in one line. Your notebook kept running.
- With a real key, the retired model produced a 4xx error whose message names the problem. This is exactly the failure that hits tutorials using old model IDs.
- `max_retries=0` makes these demos fail fast. In real code keep retries on: they help with 429 and 5xx errors, not with 401 or a retired model.

> **Coming next:** `llama-3.3-70b-versatile` is not an obscure ID. Official quickstarts still used it in October 2026, after its shutdown. How do you find out that an ID is retired *before* your code breaks? Lesson 00.06 answers that: it checks official docs pages against Groq's deprecation list, automatically.

---
## ⚠️ Pitfalls

| Symptom | Cause | Fix |
|---|---|---|
| `ModuleNotFoundError: No module named 'langchain_groq'` | Installed in another env, or kernel not restarted after an upgrade | Step 0 check; run the `%pip` cell; restart the kernel |
| `GroqError: The api_key client option must be set` | `GROQ_API_KEY` missing in this kernel | Lesson 00.04, then restart the kernel |
| Answer is cut off, `finish_reason == "length"` | Output limit reached | Ask for shorter answers or raise `max_tokens` |
| The answer is empty but tokens were used | Reasoning consumed the output budget | Lower `reasoning_effort` or raise `max_tokens` |
| 429 errors in a loop | Free-plan TPM/RPM (Lesson 00.07) | Fewer, smaller calls; wait between calls |

## ⚠️ Production notes

> ⚠️ **Production note: one model, no fallback.**
> This notebook calls a single model and simply reports errors. In production: keep the model ID pinned in config, define a tested fallback model (model fallback middleware, Lesson 04.02), retry 429/5xx with exponential backoff and respect `retry-after` (Lessons 01.06 and 12.02), and alert on rising 4xx/5xx rates.

> ⚠️ **Production note: checking model availability by hand.**
> Part 1 runs only when you open the notebook. In production: run a scheduled job (or a CI check, Module 10) that compares configured model IDs with the live Models API and opens an alert before a shutdown date. `tests/integration/test_groq_live.py` is a first version: run it with `python -m pytest -m integration`.

## 🏋️ Exercises

1. **(Easy)** Change `reasoning_effort` to `"medium"` and rerun Parts 2–3. How do `output_tokens` and the reasoning token count change?
2. **(Easy)** Call `settings.model_reasoning` (`openai/gpt-oss-120b`) with `init_chat_model` and the same question. Compare latency (`%%time`) and cost.
3. **(Medium)** Write `ask(question: str, model_id: str) -> dict` that returns `{"model", "answer", "seconds", "input_tokens", "output_tokens"}`. Use `time.perf_counter()`. You'll reuse it in Project 00.
4. **(Stretch)** Groq also offers `include_reasoning=False` for `gpt-oss` models ([reasoning docs](https://console.groq.com/docs/reasoning)). Pass it through `model_kwargs={"include_reasoning": False}` and check whether `reasoning_content` disappears.

<details>
<summary><b>Show solutions</b></summary>

**3.**
```python
import time

def ask(question: str, model_id: str) -> dict:
    """Ask one question to one Groq model and return timing and token usage."""
    model = init_chat_model(f"groq:{model_id}", temperature=0, reasoning_effort="low", timeout=60)
    start = time.perf_counter()
    message = model.invoke(question)
    seconds = time.perf_counter() - start
    usage = message.usage_metadata or {}
    return {"model": model_id, "answer": message.text, "seconds": round(seconds, 2),
            "input_tokens": usage.get("input_tokens"), "output_tokens": usage.get("output_tokens")}

if HAVE_GROQ_KEY:
    print(ask("Name one benefit of LangGraph.", settings.model_fast))
```

**4.**
```python
quiet = init_chat_model(f"groq:{settings.model_fast}", model_kwargs={"include_reasoning": False})
print(quiet.invoke("2+2?").additional_kwargs.get("reasoning_content"))   # Expect None
```
(Don't combine `include_reasoning` with `reasoning_format`: Groq says they are mutually exclusive.)
</details>

## ✅ Checkpoint

- [ ] The version cell shows `langchain 1.x` and `langchain-groq 1.x` in `lc-mastery`.
- [ ] Part 1 lists live models and marks both configured models ✅ live.
- [ ] Part 2 printed an answer; Part 3 showed token usage and cost.
- [ ] Part 5 caught the fake-key 401 without crashing.
- [ ] **Restart Kernel and Run All Cells** passes; then clear outputs and save.

## 🔎 Docs drill (5 min)

What are `ChatGroq`'s defaults for `temperature`, `max_retries` and the request timeout? Find them in the reference page, in `ChatGroq.model_fields`, and in the source, then reconcile and log the version you checked.

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. `{k: ChatGroq.model_fields[k].default for k in ("temperature", "max_retries", "request_timeout")}`.
2. Source (`inspect.getsourcefile(ChatGroq)`): the same values; `request_timeout` has the alias `timeout`.
3. Reference page: lists the parameters but not every default, and the generic Models page quotes a different `max_retries` for models in general. Trust the installed source for **your** version.
4. Answer (langchain-groq 1.1.3): temperature `0.7`, `max_retries=2`, timeout `None` (no client-side limit). Log `importlib.metadata.version("langchain-groq")` with it.

</details>

## 🔎 Research checkpoint

Part 1 printed a `context_window` for every live model. Open the [Groq models page](https://console.groq.com/docs/models) and compare its context window for `openai/gpt-oss-20b` with the number your account returned. Do they agree? Note the answer in [`docs/research-log.md`](../../docs/research-log.md). When the docs and the API disagree, which one wins? Lesson 00.06 ranks the sources.

## Summary

- `%pip install` puts packages into the kernel's environment; verify versions right after.
- The Models API is the source of truth for model IDs on *your* account.
- `init_chat_model("groq:<model>")` is the course default; `ChatGroq` gives Groq-specific options.
- Every `AIMessage` carries `.text`, `response_metadata`, and standardized `usage_metadata`.
- Errors are information: catch them, explain them, keep the notebook running.

**Next:** [00.06 · Researching like a pro](00-06-research-skills.ipynb): why that retired model ID was still in official examples, and how to catch it.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 00.06 · Researching like a pro: docs, llms.txt, changelogs, model cards** → [`00-06-research-skills.ipynb`](00-06-research-skills.ipynb).

[Module 00 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)